# Answers 01 Forces

In [ ]:
%pip install -q numpy

# Question

**Given the forces in the bars, where is ${\color{#d62828} n_0}$?**

Fixed nodes:
$$ \color{#2a6fdb}{n_1 \, (0, 0, 0) \text{ m}} $$
$$ \color{#2a6fdb}{n_2 \, (5, 0, 3) \text{ m}} $$
$$ \color{#2a6fdb}{n_3 \, (0, 7, 3) \text{ m}} $$
$$ \color{#2a6fdb}{n_4 \, (7, 5, 0) \text{ m}} $$

Load on the free node:
$$ \color{#2e9e4a}{p \, (0, 0, -5) \text{ kN}} $$

The free node ${\color{#d62828} n_0}$ is connected to four fixed nodes ${\color{#2a6fdb} n_1 \ldots n_4}$ by bars $e_0, e_1, e_2, e_3$:

$$
\newcommand{\cell}[1]{\hphantom{-}\mathclap{#1}\hphantom{-}}
\begin{array}{rcc}
 & & \begin{matrix} \cell{n_0} & \cell{n_1} & \cell{n_2} & \cell{n_3} & \cell{n_4} \end{matrix} \\[2pt]
C = & \begin{matrix} e_0 \\ e_1 \\ e_2 \\ e_3 \end{matrix} &
\left[ \;
\begin{matrix}
\cell{1} & \cell{-1} & \cell{0} & \cell{0} & \cell{0} \\
\cell{1} & \cell{0} & \cell{-1} & \cell{0} & \cell{0} \\
\cell{1} & \cell{0} & \cell{0} & \cell{-1} & \cell{0} \\
\cell{1} & \cell{0} & \cell{0} & \cell{0} & \cell{-1}
\end{matrix}
\; \right]
\end{array}
$$

The force in every bar:

$$ f = [1, 1, 1, 1] \text{ kN} $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_one_node.svg"
       style="width:100%; height:auto;"
       alt="Free node n0 with bars e0 to e3 to fixed nodes n1 to n4 and load p" />
  <p style="text-align:center;">
    <em>The free node n<sub>0</sub> connected by bars e<sub>0</sub> to e<sub>3</sub> to the fixed nodes n<sub>1</sub> to n<sub>4</sub>, with load p.</em>
  </p>
</div>

- Determine ${\color{#d62828} n_0}$.
- Calculate the sum of forces in node ${\color{#d62828} n_0}$.
- What do you observe?

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">


# Network

In [1]:
import numpy as np

# connectivity: rows = bars e0..e3, columns = nodes n0..n4
C = np.array([
    [1, -1,  0,  0,  0],
    [1,  0, -1,  0,  0],
    [1,  0,  0, -1,  0],
    [1,  0,  0,  0, -1],
])

free  = [0]            # n0
fixed = [1, 2, 3, 4]   # n1..n4

C_N = C[:, free]
C_F = C[:, fixed]

# fixed node coordinates (n1..n4)
x_F = np.array([0, 5, 0, 7])
y_F = np.array([0, 0, 7, 5])
z_F = np.array([0, 3, 3, 0])

# loads on the free node
p_x = np.array([0])
p_y = np.array([0])
p_z = np.array([-5])

# bar forces, one per bar
f = np.array([1, 1, 1, 1])

# Iteration

The linear solve needs force densities, but only the forces are given. The lengths $l$ are unknown until ${\color{#d62828} n_0}$ is known:

$$ q = \frac{\text{force in the edge}}{\text{edge length}} = \frac{f}{l} $$

So: guess $q$, solve, measure the lengths, correct $q = f / l$, repeat.

$$
\begin{aligned}
x_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_x - C_N^{T}\,Q\,C_F\,x_F\right) \\
y_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_y - C_N^{T}\,Q\,C_F\,y_F\right) \\
z_N &= \left(C_N^{T}\,Q\,C_N\right)^{-1}\left(p_z - C_N^{T}\,Q\,C_F\,z_F\right)
\end{aligned}
$$

In [2]:
q = np.array([1, 1, 1, 1])     # first guess

for step in range(10):
    Q = np.diag(q)

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_N = np.linalg.solve(A, p_x - C_N.T @ Q @ C_F @ x_F)
    y_N = np.linalg.solve(A, p_y - C_N.T @ Q @ C_F @ y_F)
    z_N = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_F)

    # bar lengths
    x = np.concatenate([x_N, x_F])
    y = np.concatenate([y_N, y_F])
    z = np.concatenate([z_N, z_F])
    l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)

    print("step", step, " n0 =", np.round([x_N[0], y_N[0], z_N[0]], 2), " bar forces =", np.round(q * l, 2))

    # correct the force densities so that the bar forces become 1 kN
    q = f / l

step 0  n0 = [3.   3.   0.25]  bar forces = [4.25 4.53 5.71 4.48]
step 1  n0 = [ 3.12  2.74 -4.46]  bar forces = [1.43 1.8  1.6  1.41]
step 2  n0 = [ 3.1   2.81 -7.77]  bar forces = [1.45 1.38 1.31 1.42]
step 3  n0 = [  3.08   2.88 -11.3 ]  bar forces = [1.37 1.3  1.27 1.35]
step 4  n0 = [  3.06   2.92 -15.38]  bar forces = [1.32 1.27 1.26 1.32]
step 5  n0 = [  3.05   2.94 -20.26]  bar forces = [1.3  1.26 1.25 1.3 ]
step 6  n0 = [  3.04   2.96 -26.23]  bar forces = [1.28 1.25 1.25 1.28]
step 7  n0 = [  3.03   2.97 -33.56]  bar forces = [1.27 1.25 1.24 1.27]
step 8  n0 = [  3.02   2.97 -42.65]  bar forces = [1.27 1.25 1.24 1.27]
step 9  n0 = [  3.02   2.98 -53.95]  bar forces = [1.26 1.25 1.24 1.26]


${\color{#d62828} n_0}$ never settles. It falls lower at every step and the bar forces never reach 1 kN.

# Sum of Forces

Each bar pulls ${\color{#d62828} n_0}$ towards its fixed node with 1 kN. Add the four pulls and the load:

$$ \sum_i f_i \, \frac{{\color{#2a6fdb} n_i} - {\color{#d62828} n_0}}{l_i} + {\color{#2e9e4a} p} $$

In [3]:
# pull of each bar on n0 = force * direction (direction = unit vector from n0 to the fixed node)
pull_x = f * -(C @ x) / l
pull_y = f * -(C @ y) / l
pull_z = f * -(C @ z) / l

sum_x = pull_x.sum() + p_x[0]
sum_y = pull_y.sum() + p_y[0]
sum_z = pull_z.sum() + p_z[0]

print("sum of forces in n0 =", np.round([sum_x, sum_y, sum_z], 2), "kN")

sum of forces in n0 = [-0.    0.   -1.01] kN


# Observation

- The sum of forces is not zero: about 1 kN is still pulling down. Four bars of 1 kN can lift 4 kN at most, the load is 5 kN. **There is no equilibrium position of ${\color{#d62828} n_0}$ for these forces.**
- With given **forces** the problem is nonlinear: the bar directions and lengths depend on the unknown ${\color{#d62828} n_0}$. It needs iterations, and a solution may not exist.
- With given **force densities** (Exercise 02) the same net is solved in one linear step.

Try `p_z = np.array([-1])` in the first cell: the loop then settles at $n_0 \, (3.14, 2.60, 0.24)$ and the sum of forces becomes zero.